# ML Distributions — A hands-on guide

A practical, runnable companion for understanding the 7 distributions you'll meet over and over in ML.

**For each distribution we do the same 4 things:**
1. Generate (or load) sample data
2. Plot the empirical distribution alongside the theoretical curve
3. Fit the distribution to data (estimate parameters from observed values)
4. Note which ML model is built on this distribution's assumptions

**Outline**
- §0 Setup
- §1 The diagnostic workflow (read me first)
- §2 Discrete: Binomial · Hypergeometric · Geometric · Poisson
- §3 Continuous: Normal · Log-normal · Exponential
- §4 Identifying an unknown distribution from raw data
- §5 Model-selection cheat sheet
- §6 Mini case study on a real dataset


## §0 Setup

In [14]:
# Core
import numpy as np
import pandas as pd
from scipy import stats

# Plotly (interactive)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = 'plotly_white'

# Seaborn — just for example datasets (tips, penguins, etc.)
import seaborn as sns

# Reproducibility
np.random.seed(42)

print("Setup complete.")


Setup complete.


### Helper functions

We'll reuse these to keep the rest of the notebook tidy.


In [15]:
def plot_discrete(samples, k_values, theoretical_pmf, title):
    """Overlay empirical bar chart with theoretical PMF for a discrete distribution."""
    counts = pd.Series(samples).value_counts(normalize=True).sort_index()
    # Reindex to ensure all k values are present (with 0 where unobserved)
    counts = counts.reindex(k_values, fill_value=0)

    fig = go.Figure()
    fig.add_bar(x=k_values, y=counts.values, name='Empirical (sampled)',
                marker_color='lightsteelblue')
    fig.add_scatter(x=k_values, y=theoretical_pmf, mode='lines+markers',
                    name='Theoretical PMF', marker_color='crimson',
                    line=dict(width=2))
    fig.update_layout(title=title, xaxis_title='k', yaxis_title='Probability',
                      height=400, bargap=0.1)
    return fig


def plot_continuous(samples, x_range, theoretical_pdf, title, fitted_label='Theoretical PDF'):
    """Overlay empirical histogram with theoretical PDF for a continuous distribution."""
    fig = go.Figure()
    fig.add_histogram(x=samples, histnorm='probability density', nbinsx=50,
                      name='Empirical (sampled)', marker_color='lightsteelblue')
    fig.add_scatter(x=x_range, y=theoretical_pdf, mode='lines',
                    name=fitted_label, line=dict(color='crimson', width=2.5))
    fig.update_layout(title=title, xaxis_title='x', yaxis_title='Density',
                      height=400)
    return fig


def qq_plot(samples, dist, title, dist_args=()):
    """Quantile-quantile plot — straight diagonal = good fit."""
    samples_sorted = np.sort(samples)
    n = len(samples_sorted)
    # Theoretical quantiles using plotting positions
    quantiles = (np.arange(1, n + 1) - 0.5) / n
    theoretical = dist.ppf(quantiles, *dist_args)

    fig = go.Figure()
    fig.add_scatter(x=theoretical, y=samples_sorted, mode='markers',
                    name='Data', marker=dict(size=4, color='steelblue'))
    lo, hi = min(theoretical.min(), samples_sorted.min()), max(theoretical.max(), samples_sorted.max())
    fig.add_scatter(x=[lo, hi], y=[lo, hi], mode='lines',
                    name='Perfect fit', line=dict(color='crimson', dash='dash'))
    fig.update_layout(title=title,
                      xaxis_title='Theoretical quantiles',
                      yaxis_title='Sample quantiles',
                      height=400, showlegend=True)
    return fig


## §1 The diagnostic workflow

Before any ML modeling, you want to *understand the data's shape*. Adopt this 4-step habit:

| Step | What you do | Why it matters |
|---|---|---|
| **1. Visualize** | Plot a histogram | The shape immediately rules out half the distributions |
| **2. Identify** | Match the shape to a candidate distribution | Constrains your modeling choices |
| **3. Fit** | Estimate the parameters from data (`scipy.stats.<dist>.fit()` or method of moments) | Now you have a working model |
| **4. Validate** | QQ plot, KS test, AIC/BIC if comparing fits | Confirms (or refutes) your guess |

You'll see this pattern repeat for every distribution below. Eventually it becomes reflex.


---
## §2 Discrete distributions

Discrete = countable outcomes (0, 1, 2, 3 …). We use **PMF** (probability mass function) — the probability of *exactly* k.


### §2.1 Binomial — "successes in n trials"

**Parameters**: `n` (number of independent trials), `p` (probability of success per trial).
**Mean**: `np` · **Variance**: `np(1-p)`
**Use when**: each observation is a count of successes in a fixed number of yes/no trials.

**ML connection** → **Logistic regression**. Every "did this prediction work?" outcome over `n` examples is Binomially distributed. The likelihood that logistic regression maximizes *is* a Binomial likelihood.

**Real example**: out of 100 emails, how many are spam? Out of 1,000 page visits, how many converted?


In [16]:
# Generate samples
n, p = 20, 0.3
samples = np.random.binomial(n, p, size=5000)

# Theoretical PMF
k = np.arange(0, n + 1)
pmf = stats.binom.pmf(k, n, p)

fig = plot_discrete(samples, k, pmf, f'Binomial(n={n}, p={p}) — 5,000 samples')
fig.show()

print(f"Empirical mean: {samples.mean():.3f}  ·  Theoretical mean (np): {n*p}")
print(f"Empirical var:  {samples.var():.3f}  ·  Theoretical var (np(1-p)): {n*p*(1-p)}")


Empirical mean: 5.976  ·  Theoretical mean (np): 6.0
Empirical var:  4.197  ·  Theoretical var (np(1-p)): 4.199999999999999


**Fitting Binomial from data**: if you know `n`, then `p_hat = mean(data) / n`. If `n` is also unknown, it's harder (rarely the case in practice).


In [17]:
# Fit p when n is known
p_hat = samples.mean() / n
print(f"True p: {p}  ·  Estimated p_hat: {p_hat:.4f}")


True p: 0.3  ·  Estimated p_hat: 0.2988


### §2.2 Hypergeometric — "Binomial without replacement"

**Parameters**: `M` (population size), `K` (number of successes in population), `n` (number of draws).
**Use when**: you're sampling from a finite pool and *not* putting items back.

The trials are no longer independent — each draw shifts the odds for the next. As `M → ∞`, Hypergeometric converges to Binomial.

**ML connection** → less common in standard models, but critical for **finite-population sampling** problems: quality control on a fixed-size batch, audit sampling, feature subset selection theory.

**Real example**: a batch of 50 chips has 15 defectives. You inspect 10. How many defectives in the sample?


In [18]:
# Population: 50 chips, 15 defective. Inspect 10.
M, K, n_draw = 50, 15, 10
samples = np.random.hypergeometric(K, M - K, n_draw, size=5000)

k = np.arange(0, n_draw + 1)
pmf = stats.hypergeom.pmf(k, M, K, n_draw)

fig = plot_discrete(samples, k, pmf, f'Hypergeometric(M={M}, K={K}, n={n_draw})')
fig.show()

# Compare against the Binomial approximation (p = K/M)
binom_pmf = stats.binom.pmf(k, n_draw, K / M)
fig2 = go.Figure()
fig2.add_scatter(x=k, y=pmf, mode='lines+markers', name='Hypergeometric (true)')
fig2.add_scatter(x=k, y=binom_pmf, mode='lines+markers', name='Binomial approx.')
fig2.update_layout(title='Hypergeometric vs Binomial approximation',
                   xaxis_title='k', yaxis_title='Probability', height=400)
fig2.show()


### §2.3 Geometric — "trials until first success"

**Parameter**: `p` (success probability per trial).
**Mean**: `1/p` · **Variance**: `(1-p)/p²`
**Shape**: always monotonically decreasing — most attempts succeed early.

**ML connection** → **churn / time-to-event modeling** in discrete time. Also appears in **reinforcement learning** as the distribution of steps until reward in a Bernoulli reward setting.

**Real example**: how many cold-call attempts before the first sale? How many login attempts before a user gets their password right?

**Note**: scipy's geometric starts at `k=1` (counting the successful trial). Some textbooks use `k=0` (counting failures before success). Be aware of the convention.


In [19]:
p = 0.25
samples = np.random.geometric(p, size=5000)

k = np.arange(1, 21)
pmf = stats.geom.pmf(k, p)

fig = plot_discrete(samples[samples <= 20], k, pmf, f'Geometric(p={p}) — trials until first success')
fig.show()

# Fit
p_hat = 1 / samples.mean()
print(f"True p: {p}  ·  Estimated p_hat: {p_hat:.4f}")


True p: 0.25  ·  Estimated p_hat: 0.2482


### §2.4 Poisson — "count of events in a window"

**Parameter**: `λ` (lambda, the average rate).
**Mean = Variance = λ**. This equality is a strong, testable assumption.
**Use when**: events occur independently at a constant average rate.

**ML connection** → **Poisson regression** (a generalized linear model) for predicting count outcomes: number of clicks, accidents, support tickets, hospital admissions. If your variance is *much larger* than the mean (overdispersion), use **Negative Binomial regression** instead.

**Real example**: customer arrivals at a store per hour, mutations per gene, goals per soccer match.


In [20]:
lam = 4.0
samples = np.random.poisson(lam, size=5000)

k = np.arange(0, 16)
pmf = stats.poisson.pmf(k, lam)

fig = plot_discrete(samples[samples <= 15], k, pmf, f'Poisson(λ={lam})')
fig.show()

# Key check: is the variance ≈ the mean?
print(f"Empirical mean: {samples.mean():.3f}")
print(f"Empirical var:  {samples.var():.3f}")
print(f"Ratio (variance/mean): {samples.var() / samples.mean():.3f}  (≈1 ⇒ Poisson is plausible)")

# Fit
lam_hat = samples.mean()
print(f"\nTrue λ: {lam}  ·  Estimated λ_hat: {lam_hat:.4f}")


Empirical mean: 4.044
Empirical var:  4.129
Ratio (variance/mean): 1.021  (≈1 ⇒ Poisson is plausible)

True λ: 4.0  ·  Estimated λ_hat: 4.0436


**Practical tip — the dispersion test**

```
ratio = variance / mean
ratio ≈ 1 → Poisson fits
ratio > 1 → overdispersed → try Negative Binomial
ratio < 1 → underdispersed → rare; consider Conway-Maxwell-Poisson
```


---
## §3 Continuous distributions

Continuous = any real value on a range. We use **PDF** (probability density function). Probabilities come from *areas under* the curve, not heights.


### §3.1 Normal — the bell curve

**Parameters**: `μ` (mean), `σ` (standard deviation).
**The 68-95-99.7 rule**: roughly 68% of values within ±1σ, 95% within ±2σ, 99.7% within ±3σ.

**Why it dominates**: the **Central Limit Theorem** — averages of many independent random things tend toward Normal, regardless of the underlying distribution. This is why measurement errors, sample means, and most "noise" are approximately Normal.

**ML connection** → assumed almost everywhere:
- **Linear regression** assumes Normal errors
- **Gaussian Naive Bayes**
- **PCA** (works best for Normal-like data)
- **Kalman filters**, **Gaussian processes**, **VAEs**
- Weight initialization in neural networks

**Real example**: heights, IQ scores, measurement errors, sample means.


In [21]:
mu, sigma = 170, 10  # adult heights in cm
samples = np.random.normal(mu, sigma, size=5000)

x = np.linspace(samples.min(), samples.max(), 200)
pdf = stats.norm.pdf(x, mu, sigma)

fig = plot_continuous(samples, x, pdf, f'Normal(μ={mu}, σ={sigma}) — adult heights (cm)')
fig.show()


In [22]:
# Verify the 68-95-99.7 rule
for k in [1, 2, 3]:
    within = np.mean(np.abs(samples - mu) <= k * sigma) * 100
    print(f"Within ±{k}σ: {within:.1f}%  (theoretical: {[68.3, 95.5, 99.7][k-1]}%)")


Within ±1σ: 69.4%  (theoretical: 68.3%)
Within ±2σ: 95.6%  (theoretical: 95.5%)
Within ±3σ: 99.7%  (theoretical: 99.7%)


In [23]:
# QQ plot — the gold-standard visual diagnostic
fig = qq_plot(samples, stats.norm, 'QQ plot vs Normal — straight line = good fit',
              dist_args=(mu, sigma))
fig.show()


In [24]:
# Fit Normal to data (estimate parameters)
mu_hat, sigma_hat = stats.norm.fit(samples)
print(f"True (μ, σ): ({mu}, {sigma})")
print(f"Fit  (μ, σ): ({mu_hat:.3f}, {sigma_hat:.3f})")

# Formal test: Shapiro-Wilk (good for small samples; KS for any size)
# H0: data is Normal — small p-value means reject Normal
ks_stat, p_value = stats.kstest(samples, 'norm', args=(mu_hat, sigma_hat))
print(f"\nKolmogorov-Smirnov test: stat={ks_stat:.4f}, p-value={p_value:.4f}")
print("(p > 0.05 ⇒ no evidence against Normal)")


True (μ, σ): (170, 10)
Fit  (μ, σ): (169.838, 9.811)

Kolmogorov-Smirnov test: stat=0.0070, p-value=0.9664
(p > 0.05 ⇒ no evidence against Normal)


### §3.2 Log-normal — Normal in log-space

**Definition**: if `log(X)` is Normal, then `X` is Log-normal. Strictly positive, right-skewed.

**When you see it**: anything produced by *multiplying* lots of small random factors:
- Income, wealth
- City populations
- File sizes
- Stock prices, returns over long horizons
- Reaction times
- Time spent on a webpage

**ML connection** → **log-transform skewed positive features** before modeling. This is one of the most common feature-engineering moves. Then linear models, neural nets, and tree-based models all behave better.

**Real example**: restaurant tips, house prices, customer lifetime value.


In [25]:
# Generate log-normal data
mu_log, sigma_log = 0, 0.6
samples = np.random.lognormal(mu_log, sigma_log, size=5000)

x = np.linspace(0.01, samples.max(), 200)
pdf = stats.lognorm.pdf(x, s=sigma_log, scale=np.exp(mu_log))

fig = plot_continuous(samples, x, pdf, f'Log-normal(μ={mu_log}, σ={sigma_log}) — raw scale')
fig.show()


In [26]:
# The magic: log-transform the data and it becomes Normal
log_samples = np.log(samples)

x_log = np.linspace(log_samples.min(), log_samples.max(), 200)
pdf_log = stats.norm.pdf(x_log, mu_log, sigma_log)

fig = plot_continuous(log_samples, x_log, pdf_log,
                      'After log-transform — now Normal!',
                      fitted_label='Normal PDF')
fig.show()


In [27]:
# Try on a real dataset — tips at restaurants
tips = sns.load_dataset('tips')
print(tips.head())

# total_bill tends to be right-skewed
fig = px.histogram(tips, x='total_bill', nbins=40,
                   title='Restaurant total_bill — raw')
fig.show()

fig = px.histogram(tips, x=np.log(tips['total_bill']), nbins=40,
                   title='Restaurant log(total_bill) — much more symmetric')
fig.show()

# Fit log-normal directly
shape, loc, scale = stats.lognorm.fit(tips['total_bill'], floc=0)
print(f"\nFit log-normal to total_bill:")
print(f"  σ (shape): {shape:.4f}")
print(f"  exp(μ) (scale): {scale:.4f}")


   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3
3       23.68  3.31    Male     No  Sun  Dinner     2
4       24.59  3.61  Female     No  Sun  Dinner     4



Fit log-normal to total_bill:
  σ (shape): 0.4380
  exp(μ) (scale): 17.9989


**Practical tip** — to test if a positive variable is log-normal, take `np.log(x)` and then do a Normality test (Shapiro / KS) on the result. If the log is Normal, the original is log-normal.


### §3.3 Exponential — "time between random events"

**Parameter**: `λ` (rate) — or equivalently `β = 1/λ` (scale, the mean time).
**Mean = 1/λ**. **Memoryless**: the wait from now is independent of how long you've already waited.

**Pairing with Poisson**: if event *counts* in a window are Poisson(λ), the *gaps between events* are Exponential(λ). They're two sides of the same process.

**ML connection** → **Survival analysis** (Cox proportional hazards, accelerated failure time models). Time-to-event modeling: time until a customer churns, a machine fails, a patient relapses.

**Real example**: time between earthquakes, time between login attempts, lifetime of a lightbulb.


In [28]:
lam = 0.5
samples = np.random.exponential(scale=1/lam, size=5000)

x = np.linspace(0, samples.max(), 200)
pdf = stats.expon.pdf(x, scale=1/lam)

fig = plot_continuous(samples, x, pdf, f'Exponential(λ={lam})')
fig.show()

# Fit
loc, scale = stats.expon.fit(samples, floc=0)
lam_hat = 1 / scale
print(f"True λ: {lam}  ·  Estimated λ_hat: {lam_hat:.4f}")


True λ: 0.5  ·  Estimated λ_hat: 0.5078


In [29]:
# Demonstrate the Poisson-Exponential pairing
# Generate event arrival times where the count per unit time is Poisson(λ=3)
lam_rate = 3.0
inter_arrival_times = np.random.exponential(scale=1/lam_rate, size=5000)
# Counts in unit windows should be Poisson(3)
event_times = np.cumsum(inter_arrival_times)
window = np.floor(event_times).astype(int)
counts_per_window = pd.Series(window).value_counts().values

print(f"Mean count per window: {counts_per_window.mean():.3f}  (should be ≈ {lam_rate})")
print(f"Var  count per window: {counts_per_window.var():.3f}  (should also be ≈ {lam_rate} for Poisson)")


Mean count per window: 3.161  (should be ≈ 3.0)
Var  count per window: 2.659  (should also be ≈ 3.0 for Poisson)


---
## §4 Identifying a distribution from raw data

You've been handed a column. What is it? Here's the practical recipe.

**Step 1 — Look at it.** Histogram first, then summary statistics.
**Step 2 — Ask the discrete/continuous question.** Are values integers and countable, or measurements?
**Step 3 — Note the shape.** Symmetric? Right-skewed? Decaying from zero? Bell-shaped?
**Step 4 — Check key signatures.**
- Mean ≈ Variance → Poisson candidate
- log(x) looks Normal → Log-normal
- Strictly positive, decaying → Exponential
- Symmetric, light tails → Normal
**Step 5 — Fit and test.** Use `stats.<dist>.fit()` and then `stats.kstest`.


In [30]:
def diagnose(data, name='data'):
    """Quick diagnostic summary for an unknown sample."""
    data = np.asarray(data)
    print(f"=== Diagnostic for '{name}' ===")
    print(f"n: {len(data)}")
    print(f"min, max: {data.min():.3f}, {data.max():.3f}")
    print(f"mean: {data.mean():.3f}")
    print(f"std:  {data.std():.3f}")
    print(f"variance / mean ratio: {data.var() / data.mean():.3f}"
          if data.mean() != 0 else "")
    print(f"skewness: {stats.skew(data):.3f}  (0=symmetric, >0=right-skewed)")
    print(f"kurtosis: {stats.kurtosis(data):.3f}  (0=Normal-like tails)")
    print(f"all integers? {np.allclose(data, np.round(data))}")
    print(f"all positive? {(data > 0).all()}")

    fig = px.histogram(x=data, nbins=50, title=f'Histogram of {name}')
    fig.show()


In [31]:
# Try the diagnostic on the tips dataset
diagnose(tips['total_bill'], 'total_bill')


=== Diagnostic for 'total_bill' ===
n: 244
min, max: 3.070, 50.810
mean: 19.786
std:  8.884
variance / mean ratio: 3.989
skewness: 1.126  (0=symmetric, >0=right-skewed)
kurtosis: 1.169  (0=Normal-like tails)
all integers? False
all positive? True


In [32]:
# Compare fits — which distribution scores best?
def compare_fits(data, candidates=('norm', 'lognorm', 'expon', 'gamma')):
    results = []
    for name in candidates:
        dist = getattr(stats, name)
        try:
            if name in ('lognorm', 'expon', 'gamma'):
                params = dist.fit(data, floc=0)
            else:
                params = dist.fit(data)
            ks_stat, p = stats.kstest(data, name, args=params)
            # Log-likelihood for AIC
            ll = np.sum(dist.logpdf(data, *params))
            k = len(params)
            aic = 2 * k - 2 * ll
            results.append({'distribution': name, 'KS_stat': ks_stat,
                            'p_value': p, 'AIC': aic, 'params': params})
        except Exception as e:
            results.append({'distribution': name, 'error': str(e)})
    return pd.DataFrame(results).sort_values('AIC')

compare_fits(tips['total_bill'].values)


,distribution,KS_stat,p_value,AIC,params
1,lognorm,0.035828,9.015085e-01,1706.024446,"(0.43797557153283817, 0, 17.99889309665678)"
3,gamma,0.065127,2.412367e-01,1710.398086,"(5.443174520114092, 0, 3.63500059567006)"
0,norm,0.118846,1.846324e-03,1762.365206,"(19.78594262295082, 8.884150577771132)"
2,expon,0.333684,1.010542e-24,1948.666198,"(0.0, 19.78594262295082)"


**How to read this table**:
- **Lower AIC = better fit** (penalizes model complexity).
- **Higher p-value in KS test = no evidence against the distribution**. (Counter-intuitive: high p is *good* here.)
- If two candidates score close, prefer the simpler one or the one with better domain justification.


---
## §5 Model-selection cheat sheet — bookmark this section

### From data type → distribution

| Your data looks like… | Likely distribution | Quick check |
|---|---|---|
| Binary outcomes (0/1) | Binomial | Count of 1s out of n |
| Counts (0, 1, 2 …) | Poisson | variance ≈ mean? |
| Counts, overdispersed | Negative Binomial | variance ≫ mean |
| Trials until first success | Geometric | Monotonically decreasing PMF |
| Counts without replacement | Hypergeometric | Sampling from finite pool |
| Continuous, symmetric bell | Normal | Skewness ≈ 0, QQ plot straight |
| Continuous, right-skewed, positive | Log-normal | log(x) looks Normal |
| Continuous, positive, decaying | Exponential | Memoryless; gaps between events |
| Continuous, bounded [0, 1] | Beta | Proportions, probabilities |
| Continuous, positive, flexible shape | Gamma | Generalization of Exponential |

### From distribution → ML model

| Distribution of target/errors | Use this model |
|---|---|
| Binomial (binary y) | **Logistic regression**, classifiers in general |
| Multinomial (categorical y) | **Softmax / multinomial logistic regression** |
| Poisson (count y) | **Poisson regression** (a GLM) |
| Negative Binomial (count, overdispersed) | **Negative Binomial regression** |
| Normal errors | **Linear regression**, ridge, lasso |
| Log-normal y | **Log-transform y**, then linear regression; or **Gamma GLM** |
| Exponential / Weibull time-to-event | **Survival models** (Cox PH, AFT) |
| Beta (proportions) | **Beta regression** |

### Feature engineering moves

| If a feature is… | Try… |
|---|---|
| Right-skewed and positive | `np.log(x)` or `np.log1p(x)` |
| Heavy-tailed | Box-Cox / Yeo-Johnson transform |
| On wildly different scales | Standardization (`StandardScaler`) |
| Counts | Sometimes `np.sqrt(x)` stabilizes variance |
| Bounded [0, 1] | Logit transform: `log(p/(1-p))` |

### Red flags

- **Linear regression on a count target** → wrong; use Poisson regression.
- **Linear regression on right-skewed `y` with no transform** → predictions will be biased; log-transform first.
- **Assuming Normal when tails are heavy** → outliers will wreck you; use robust models or t-distribution.
- **Variance much bigger than mean on count data** → don't use Poisson; use Negative Binomial.


---
## §6 Mini case study — applying the whole workflow

Let's walk through a realistic mini-analysis on the **penguins** dataset. We'll examine `flipper_length_mm` and decide how to model it.


In [33]:
penguins = sns.load_dataset('penguins').dropna(subset=['flipper_length_mm'])
print(penguins[['species', 'flipper_length_mm']].head())
print(f"\nShape: {penguins.shape}")


  species  flipper_length_mm
0  Adelie              181.0
1  Adelie              186.0
2  Adelie              195.0
4  Adelie              193.0
5  Adelie              190.0

Shape: (342, 7)


In [34]:
# Step 1: Visualize
fig = px.histogram(penguins, x='flipper_length_mm', nbins=40,
                   title='Flipper length distribution (all species)')
fig.show()


Hmm — that looks **bimodal**, not Normal! This is a hint that the population is a *mixture* of subgroups. Color by species:

In [35]:
fig = px.histogram(penguins, x='flipper_length_mm', color='species', nbins=40,
                   barmode='overlay', opacity=0.6,
                   title='Flipper length by species')
fig.show()


Within each species, the distribution looks roughly Normal. The bimodality came from mixing species. This is a common pattern: **always check whether a "weird" distribution is actually a mixture of well-behaved subgroups.**

Let's confirm Normality within one species.

In [36]:
adelie = penguins[penguins['species'] == 'Adelie']['flipper_length_mm'].values

diagnose(adelie, 'Adelie flipper length')

# Fit Normal
mu_hat, sigma_hat = stats.norm.fit(adelie)
print(f"\nFitted Normal: μ={mu_hat:.2f}, σ={sigma_hat:.2f}")

# QQ plot
fig = qq_plot(adelie, stats.norm, 'QQ plot — Adelie flipper length vs Normal',
              dist_args=(mu_hat, sigma_hat))
fig.show()

# Formal test
ks_stat, p = stats.kstest(adelie, 'norm', args=(mu_hat, sigma_hat))
print(f"KS test p-value: {p:.4f}  ({'no evidence against Normal' if p > 0.05 else 'reject Normal'})")


=== Diagnostic for 'Adelie flipper length' ===
n: 151
min, max: 172.000, 210.000
mean: 189.954
std:  6.518
variance / mean ratio: 0.224
skewness: 0.086  (0=symmetric, >0=right-skewed)
kurtosis: 0.282  (0=Normal-like tails)
all integers? True
all positive? True



Fitted Normal: μ=189.95, σ=6.52


KS test p-value: 0.3891  (no evidence against Normal)


**Take-aways from this case study**

1. The raw `flipper_length_mm` is bimodal → a single distribution doesn't fit.
2. Conditioning on `species` reveals approximately Normal subgroups.
3. **Modeling implication**: rather than modeling `flipper_length_mm` alone, include species as a predictor — a linear regression with species as a categorical feature would capture the structure.
4. If you'd plowed ahead and assumed Normal on the full data, your prediction intervals and any z-score-based outlier detection would have been off.

**This is the entire point**: distribution diagnosis isn't a math exercise. It changes how you model.


---
## §7 Next steps and further reading

- **Practice**: try the workflow on a Kaggle dataset of your choice. Pick a column, diagnose it, model it.
- **GLMs**: read about **generalized linear models** — they unify linear, logistic, Poisson, and Gamma regression under one framework. `statsmodels` is the Python library to use.
- **Survival analysis**: `lifelines` library — applies Exponential / Weibull / Cox models.
- **Mixture models**: `sklearn.mixture.GaussianMixture` — for when your data is a mixture of Normals (like the penguin flippers).
- **Bayesian view**: every distribution we covered is also a **prior** or **likelihood** building block in Bayesian models (`pymc`, `numpyro`).

**The mental shift to make** is: every ML algorithm has distributional assumptions baked into it. Knowing the distributions = knowing which algorithm to reach for, *and why*.
